In [ ]:
import os
for f in sorted(os.listdir()):
    print(f)

.config
Indian Startups - Funding  Investors Data April 2022.csv
Indian Startups - Funding  Investors Data February 2022.csv
Indian Startups - Funding  Investors Data January 2022.csv
Indian Startups - Funding  Investors Data March 2022.csv
Indian Startups - Funding  Investors Data May 2022.csv
sample_data
startup_funding.csv
startup_funding2018.csv
startup_funding2019.csv
startup_funding2020.csv
startup_funding2021.csv


In [ ]:
import pandas as pd, glob

frames = []

# 1. 2015-2017 from main Kaggle file
d = pd.read_csv('startup_funding.csv', dtype=str)
d['Year'] = d['Date dd/mm/yyyy'].str.extract(r'(\d{4})')[0]
d = d[d['Year'].isin(['2015','2016','2017'])]          # drop overlapping years
d = d.rename(columns={'Date dd/mm/yyyy':'Date','Startup Name':'Startup',
     'Industry Vertical':'Industry','City  Location':'City',
     'Investors Name':'Investors','InvestmentnType':'Stage','Amount in USD':'Amount'})
d['Source_File'] = 'startup_funding.csv (Kaggle, sudalairajkumar)'
frames.append(d)

# 2. 2018 (different column names)
d = pd.read_csv('startup_funding2018.csv', dtype=str)
d = d.rename(columns={'Company Name':'Startup','Location':'City',
                      'Round/Series':'Stage','About Company':'SubVertical'})
d['Year'] = '2018'; d['Source_File'] = 'startup_funding2018.csv (GitHub)'
frames.append(d)

# 3. 2019, 2020, 2021 (same format)
for y in ['2019','2020','2021']:
    d = pd.read_csv(f'startup_funding{y}.csv', dtype=str)
    d = d.rename(columns={'Company/Brand':'Startup','HeadQuarter':'City','Sector':'Industry',
                          'Investor':'Investors','Amount($)':'Amount','What it does':'SubVertical'})
    d['Year'] = y; d['Source_File'] = f'startup_funding{y}.csv (GitHub)'
    frames.append(d)

# 4. 2022 monthly files (finds them by month name)
for m in ['January','February','March','April','May']:
    f = [x for x in glob.glob('*.csv') if m in x and '2022' in x][0]
    d = pd.read_csv(f, dtype=str, encoding='latin1', on_bad_lines='skip').dropna(how='all')
    d = d.rename(columns={'Company Name':'Startup','Location':'City','Sector':'Industry',
                          'Investor':'Investors','What it does':'SubVertical'})
    d['Year'] = '2022'; d['Date'] = m + ' 2022'; d['Source_File'] = f'{m} 2022 (Kaggle)'
    frames.append(d)

# 5. Stack everything and keep common columns
cols = ['Year','Date','Startup','Industry','SubVertical','City','Investors','Stage','Amount','Source_File']
raw = pd.concat(frames, ignore_index=True).reindex(columns=cols)

print(raw.shape)                       # should be (5922, 10)
print(raw['Year'].value_counts().sort_index())
raw.to_csv('indian_startup_funding_raw.csv', index=False, encoding='utf-8-sig')

(5922, 10)
Year
2015     935
2016     993
2017     687
2018     526
2019      89
2020    1055
2021    1209
2022     428
Name: count, dtype: int64


In [ ]:
import pandas as pd
import re

df = pd.read_csv('indian_startup_funding_raw.csv', dtype=str)
print(df.shape)

# See 30 random Amount values to understand the problem
for v in df['Amount'].sample(30, random_state=1): print(v)

(5922, 10)
4000000
Undisclosed
20,00,000
nan
nan
$12,000,000
$1000000
nan
$10000000
$10,000,000
nan
24,00,000
nan
2,00,000
nan
48,50,000
5000000
nan
15,00,000
17,00,000
nan
nan
$41,900,000
5,00,000
nan
$1,000,000
22,50,00,000
$21,000,000
50,00,000
nan


In [ ]:
def clean_amount(x):
    if pd.isna(x):
        return None
    s = str(x).strip().lower()

    # 1. Undisclosed / unknown -> missing
    if 'undisclosed' in s or 'undiclsosed' in s or s in ['', 'n/a', 'unknown', '-']:
        return None

    # 2. Ranges like "$800,000,000 to $850,000,000" -> take first value
    if ' to ' in s:
        s = s.split(' to ')[0]

    # 3. Is it in rupees?
    is_rupee = '₹' in s

    # 4. Odd format like "$468.200,000" (dot used as comma) -> remove dots
    if re.search(r'\d\.\d{3},', s):
        s = s.replace('.', '')

    # 5. Keep only digits and decimal point
    s = re.sub(r'[^0-9.]', '', s)
    if s == '' or s.count('.') > 1:
        return None

    value = float(s)

    # 6. Convert rupees to dollars (approx Rs 70 = $1 in 2018)
    if is_rupee:
        value = value / 70

    return value

df['Amount_USD'] = df['Amount'].apply(clean_amount)

In [ ]:
print("Rows with amount:", df['Amount_USD'].notna().sum())
print("Rows missing amount:", df['Amount_USD'].isna().sum())

# Before vs after, side by side
print(df[['Amount', 'Amount_USD']].sample(15, random_state=2))

# Biggest values (to find outliers)
print(df.sort_values('Amount_USD', ascending=False)[['Year','Startup','Amount','Amount_USD']].head(15))

# Smallest values (suspiciously tiny)
print(df.sort_values('Amount_USD')[['Year','Startup','Amount','Amount_USD']].head(15))

Rows with amount: 4402
Rows missing amount: 1520
            Amount   Amount_USD
1069     53,00,000    5300000.0
1277     10,00,000    1000000.0
4210   Undisclosed          NaN
5486       $500000     500000.0
1691   2,00,00,000   20000000.0
4451    $450000000  450000000.0
4841       $300000     300000.0
257      10,00,000    1000000.0
1962           NaN          NaN
4096   Undisclosed          NaN
5891    $7,000,000    7000000.0
1884     20,00,000    2000000.0
5424       $900000     900000.0
3575  $1,90,67,328   19067328.0
1696   2,00,00,000   20000000.0
      Year                       Startup                 Amount    Amount_USD
3695  2020                  True Balance  $887,000, $23,000,000  8.870002e+13
4358  2021               Alteria Capital       $150,000,000,000  1.500000e+11
3510  2020  Reliance Retail Ventures Ltd        $70,000,000,000  7.000000e+10
5818  2022                         Accel        $18,300,000,000  1.830000e+10
5822  2022                           Ola         

In [ ]:
# 0. Start fresh from the text columns (drop old Amount_USD)
df = df.drop(columns=['Amount_USD'], errors='ignore')

# 1. Remove junk text '\xc2\xa0' from all columns
junk = r'\\+xc2\\+xa0'
print("Rows with junk text:", df.apply(lambda c: c.str.contains(junk, regex=True, na=False)).any(axis=1).sum())
df = df.replace(junk, '', regex=True)
for col in df.columns:
    df[col] = df[col].str.strip()

# 2. Improved cleaning function
def clean_amount(x):
    if pd.isna(x):
        return None
    s = str(x).strip().lower()
    if any(w in s for w in ['undisclosed', 'undiclsosed', 'n/a', 'unknown']) or s in ['', '-']:
        return None
    if ' to ' in s:                        # range -> first value
        s = s.split(' to ')[0]
    if ', $' in s:                         # two amounts -> add them
        return sum(clean_amount(p) or 0 for p in s.split(', $'))
    is_rupee = '₹' in s
    if re.search(r'\d\.\d{3},', s):        # "468.200,000" style
        s = s.replace('.', '')
    s = re.sub(r'[^0-9.]', '', s)
    if s == '' or s.count('.') > 1:
        return None
    value = float(s)
    return value / 70 if is_rupee else value

df['Amount_USD'] = df['Amount'].apply(clean_amount)

# 3. Remove April 2022 "Multiples" rows (cumulative totals, not single deals)
before = len(df)
april = df['Source_File'].str.contains('April 2022')
df = df[~(april & (df['Investors'] == 'Multiples'))]
print("Removed 'Multiples' rows:", before - len(df))

# 4. 2022 amounts written in millions ($84.2 -> 84,200,000)
mask = (df['Year'] == '2022') & (df['Amount_USD'] < 1000)
print("Rescaled 2022 rows:", mask.sum())
df.loc[mask, 'Amount_USD'] = df.loc[mask, 'Amount_USD'] * 1_000_000

# 5. Impossible / non-startup values -> missing
april = df['Source_File'].str.contains('April 2022')
bad = df['Startup'].isin(['Alteria Capital', 'Reliance Retail Ventures Ltd', 'Snowflake', 'Peel Works']) \
      | (april & df['Startup'].isin(['Ola', 'Mahindra Logistics', 'Pine Labs']))
print("Set to missing:", bad.sum())
df.loc[bad, 'Amount_USD'] = None

# Check
print("Rows:", len(df), "| With amount:", df['Amount_USD'].notna().sum())
print("Total funding ($ billion):", round(df['Amount_USD'].sum() / 1e9, 1))
print(df.sort_values('Amount_USD', ascending=False)[['Year','Startup','Amount_USD']].head(10))

Rows with junk text: 87
Removed 'Multiples' rows: 31
Rescaled 2022 rows: 7
Set to missing: 8
Rows: 5891 | With amount: 4361
Total funding ($ billion): 97.8
      Year                   Startup    Amount_USD
223   2017                  Flipkart  2.500000e+09
3558  2020                  Reliance  2.200000e+09
402   2017                     Paytm  1.400000e+09
538   2017                  Flipkart  1.400000e+09
4631  2021          VerSe Innovation  1.000000e+09
2716  2018                    Swiggy  1.000000e+09
5220  2021              Dream Sports  8.400000e+08
4338  2021                    Swiggy  8.000000e+08
3702  2020                  Eruditus  8.000000e+08
3532  2020  Reliance Retail Ventures  8.000000e+08


In [ ]:
print("Unique city values:", df['City'].nunique())
print(df['City'].value_counts().head(40))

Unique city values: 242
City
Bangalore                           1578
Mumbai                               934
New Delhi                            622
Gurgaon                              342
Gurugram                             239
Chennai                              193
Pune                                 186
Noida                                167
Hyderabad                            160
Bangalore, Karnataka, India          102
Mumbai, Maharashtra, India            94
Delhi                                 85
Ahmedabad                             64
Bengaluru, Karnataka, India           55
Gurgaon, Haryana, India               52
Jaipur                                51
New Delhi, Delhi, India               51
Kolkata                               38
Indore                                21
Pune, Maharashtra, India              20
Chandigarh                            20
Chennai, Tamil Nadu, India            19
Haryana                               19
Hyderabad, Andhra Pradesh, I

In [ ]:
# 1. Keep only the part before the first comma, remove extra spaces
df['City_Clean'] = df['City'].str.split(',').str[0].str.strip()

# 2. Merge different spellings of the same city
city_map = {
    'Bangalore': 'Bengaluru', 'Banglore': 'Bengaluru', 'Bengaluru': 'Bengaluru',
    'Benglaru': 'Bengaluru', 'Bangalore City': 'Bengaluru', 'Kormangala': 'Bengaluru',
    'Gurgaon': 'Gurugram', 'Gurugram': 'Gurugram',
    'New Delhi': 'Delhi', 'Delhi': 'Delhi', 'New delhi': 'Delhi',
    'Bombay': 'Mumbai', 'Mumba': 'Mumbai', 'Andheri': 'Mumbai', 'Powai': 'Mumbai', 'Chembur': 'Mumbai',
    'Ahemdabad': 'Ahmedabad', 'Ahemadabad': 'Ahmedabad', 'Ahmadabad': 'Ahmedabad',
    'Cochin': 'Kochi', 'Kochin': 'Kochi',
    'Taramani': 'Chennai', 'Secunderabad': 'Hyderabad', 'Hyderebad': 'Hyderabad',
}
df['City_Clean'] = df['City_Clean'].replace(city_map)

# 3. Foreign cities -> 'Outside India'
foreign = ['Singapore', 'San Francisco', 'Palo Alto', 'California', 'New York', 'Newyork',
           'London', 'Santa Clara', 'San Jose', 'Delaware', 'Oakland', 'Tokyo', 'Amsterdam',
           'Petaluma', 'Maryland', 'Massachusetts', 'Israel', 'Parsippany', 'Chicago',
           'Santa Monica', 'Nairobi', 'Outside India', 'Vitznau']
df.loc[df['City_Clean'].isin(foreign), 'City_Clean'] = 'Outside India'

# 4. Empty -> 'Unknown'
df['City_Clean'] = df['City_Clean'].fillna('Unknown').replace('', 'Unknown')

print("Unique cities before:", df['City'].nunique(), "| after:", df['City_Clean'].nunique())
print(df['City_Clean'].value_counts().head(20))

Unique cities before: 242 | after: 167
City_Clean
Bengaluru        1770
Mumbai           1034
Delhi             775
Gurugram          633
Unknown           294
Chennai           212
Pune              206
Noida             182
Hyderabad         180
Ahmedabad          77
Jaipur             63
Outside India      53
Kolkata            44
Haryana            30
Indore             25
Chandigarh         21
Thane              16
Kochi              15
Goa                14
Vadodara           11
Name: count, dtype: int64


In [ ]:
print("Unique stage values:", df['Stage'].nunique())
print(df['Stage'].value_counts().head(50))

Unique stage values: 96
Stage
Seed Funding                1346
Private Equity              1214
Seed                         679
Series A                     361
Pre-series A                 248
Series B                     160
Series C                     131
Seed Round                    69
Pre-seed                      66
Series D                      63
Pre series A                  62
Debt                          45
Angel                         38
Venture - Series Unknown      37
Series E                      34
Seed\\nFunding                30
undisclosed                   29
Seed/Angel Funding            20
Series F                      17
Pre-series B                  15
Debt Financing                13
Pre Series A                  12
Debt Funding                  11
Seed round                    10
Corporate Round                8
Pre-Seed                       8
Bridge                         8
Series H                       5
Angel Round                    4
Pre-series   

In [ ]:
def stage_group(x):
    if pd.isna(x):
        return 'Unknown'
    s = str(x).lower().replace('\\n', ' ').replace('-', ' ').replace('/', ' ')
    s = ' '.join(s.split())                     # remove extra spaces

    if 'pre seed' in s or 'preseed' in s:
        return 'Pre-Seed'
    if 'pre series' in s or 'pre seies' in s:
        return 'Pre-Series A/B'
    if 'seed' in s or 'angel' in s:
        return 'Seed/Angel'
    m = re.search(r'series\s*([a-j])\b', s)     # Series A, B, C ... J
    if m:
        letter = m.group(1).upper()
        return f'Series {letter}' if letter in 'ABC' else 'Series D+'
    if 'private equity' in s or s == 'pe':
        return 'Private Equity'
    if 'debt' in s:
        return 'Debt'
    return 'Other/Unknown'                      # undisclosed, grants, bridge, $ amounts typed by mistake...

df['Stage_Clean'] = df['Stage'].apply(stage_group)
print("Unique stages before:", df['Stage'].nunique(), "| after:", df['Stage_Clean'].nunique())
print(df['Stage_Clean'].value_counts())

Unique stages before: 96 | after: 11
Stage_Clean
Seed/Angel        2207
Private Equity    1215
Unknown           1047
Series A           365
Pre-Series A/B     354
Series B           161
Series C           132
Other/Unknown      130
Series D+          126
Pre-Seed            83
Debt                71
Name: count, dtype: int64


In [ ]:
print("Unique industry values before:", df['Industry'].nunique())

sectors = [   # (sector name, keywords) - checked in this order
    ('AgriTech',            r'agri|farm|dairy|fisher|aqua'),
    ('EdTech',              r'edu|e-tech|learn|tutor|school|skill|course|exam'),
    ('HealthTech',          r'health|medic|pharma|hospital|wellness|fitness|diagnos|biotech|care\b|nutrition|hygiene'),
    ('FinTech',             r'fin|payment|lend|bank|insur|credit|wealth|invest|nbfc|loan|crypto|money|capital'),
    ('Food & Beverage',     r'food|beverage|restaurant|grocery|kitchen|snack|coffee|tea\b'),
    ('E-commerce & Retail', r'commerce|retail|marketplace|shopping|d2c|fashion|apparel|consumer goods|cosmetic|beauty|jewel'),
    ('Travel & Hospitality',r'travel|hospitality|hotel|tourism|accomodation|accommodation|co-living|coworking|co-working'),
    ('Logistics & Mobility',r'logistic|supply chain|delivery|transport|mobility|automo|\bev\b|electric vehicle|vehicle|rental'),
    ('Real Estate',         r'real estate|proptech|construction|housing|interior|home'),
    ('Media & Entertainment',r'media|gaming|game|entertainment|content|social|video|music|news|sports|ott'),
    ('HR & Jobs',           r'\bhr\b|recruit|job|staffing|human resource'),
    ('Consumer Internet',   r'consumer internet'),
    ('Tech & SaaS',         r'saas|software|tech|\bai\b|\bit\b|analytics|cloud|data|internet|automation|robot|iot|deep'),
]

def sector_group(x):
    if pd.isna(x):
        return 'Unknown'
    s = str(x).lower()
    for name, pattern in sectors:
        if re.search(pattern, s):
            return name
    return 'Other'

df['Industry_Clean'] = df['Industry'].apply(sector_group)
print("Unique industries after:", df['Industry_Clean'].nunique())
print(df['Industry_Clean'].value_counts())

# Check what landed in 'Other' - to decide if more keywords are needed
print(df.loc[df['Industry_Clean'] == 'Other', 'Industry'].value_counts().head(25))

Unique industry values before: 1675
Unique industries after: 15
Industry_Clean
Tech & SaaS              1223
Consumer Internet         848
E-commerce & Retail       721
FinTech                   611
Other                     594
HealthTech                453
Logistics & Mobility      312
Food & Beverage           262
Media & Entertainment     234
Unknown                   190
EdTech                    167
AgriTech                   98
Real Estate                71
Travel & Hospitality       65
HR & Jobs                  42
Name: count, dtype: int64
Industry
—                                   30
Consumer Services                    7
Manufacturing                        7
Renewables & Environment             7
Consulting                           7
Renewable Energy                     7
FMCG                                 7
Marketing & Advertising              6
Others                               6
B2B                                  6
Artificial Intelligence              6
Furnitu

In [ ]:
df['Startup_key'] = df['Startup'].str.lower().str.strip()
before = len(df)
df = df.drop_duplicates(subset=['Startup_key', 'Year', 'Amount_USD', 'Stage_Clean', 'Investors'])
print("Duplicates removed:", before - len(df))
print("Rows now:", len(df))
df = df.drop(columns=['Startup_key'])

Duplicates removed: 1
Rows now: 5857


In [ ]:
final_cols = ['Year', 'Date', 'Startup', 'Industry_Clean', 'City_Clean', 'Investors',
              'Stage_Clean', 'Amount_USD', 'Industry', 'City', 'Stage', 'Amount', 'Source_File']
clean = df[final_cols].rename(columns={'Industry_Clean': 'Sector', 'City_Clean': 'City_Std',
                                       'Stage_Clean': 'Stage_Std'})
clean.to_csv('startup_funding_clean.csv', index=False, encoding='utf-8-sig')
print(clean.shape)
print(clean.head())

(5857, 13)
   Year        Date        Startup       Sector   City_Std  \
0  2017  01/12/2017  Strata Enviro  Tech & SaaS     Mumbai   
1  2017  05/12/2017          Dunzo  Tech & SaaS  Bengaluru   
2  2017  05/12/2017   Tata Housing  Real Estate     Mumbai   
3  2017  05/12/2017     HandyTrain  Tech & SaaS        Goa   
4  2017  05/12/2017         Mobycy  Tech & SaaS   Gurugram   

               Investors       Stage_Std  Amount_USD     Industry       City  \
0            Scale Minds  Private Equity         NaN   Technology     Mumbai   
1                 Google  Private Equity  12300000.0   Technology  Bangalore   
2                  Hines  Private Equity  23000000.0  Real Estate     Mumbai   
3            NB Ventures  Private Equity   1000000.0   Technology        Goa   
4  Undisclosed Investors      Seed/Angel    500000.0   Technology    Gurgaon   

                Stage       Amount  \
0      Private Equity          NaN   
1      Private Equity  1,23,00,000   
2      Private Equity

In [ ]:
# 1. Merge the two unknown stage groups
df['Stage_Clean'] = df['Stage_Clean'].replace({'Other/Unknown': 'Unknown/Other', 'Unknown': 'Unknown/Other'})

# 2. Extra sector groups (inserted before the general 'Tech & SaaS')
extra = [
    ('CleanTech & Energy',      r'energy|solar|renewable|environment|clean|recycl|waste|battery'),
    ('Manufacturing & Consumer Goods', r'manufactur|electronic|fmcg|furniture|textile|consumer goods|wine|spirits|hardware'),
    ('Business Services',       r'marketing|advertis|consult|b2b|design|legal|service|coaching|training'),
]
sectors = [s for s in sectors if s[0] != 'Tech & SaaS'] + extra + [
    ('Tech & SaaS', r'saas|software|tech|\bai\b|artificial intelligence|\bit\b|analytics|cloud|data|internet|automation|robot|iot|deep|blockchain|drone|telecom'),
]

def sector_group(x):
    if pd.isna(x) or str(x).strip() in ['—', '-', '']:
        return 'Unknown'
    s = str(x).lower()
    for name, pattern in sectors:
        if re.search(pattern, s):
            return name
    return 'Other'

df['Industry_Clean'] = df['Industry'].apply(sector_group)

print(df['Stage_Clean'].value_counts())
print("\nIndustries:", df['Industry_Clean'].nunique())
print(df['Industry_Clean'].value_counts())

Stage_Clean
Seed/Angel        2196
Private Equity    1214
Unknown/Other     1165
Series A           363
Pre-Series A/B     350
Series B           161
Series C           131
Series D+          124
Pre-Seed            83
Debt                71
Name: count, dtype: int64

Industries: 18
Industry_Clean
Tech & SaaS                       1134
Consumer Internet                  848
E-commerce & Retail                716
FinTech                            604
HealthTech                         450
Other                              326
Logistics & Mobility               309
Food & Beverage                    260
Media & Entertainment              231
Unknown                            219
Business Services                  202
EdTech                             166
AgriTech                            98
Real Estate                         71
Travel & Hospitality                65
CleanTech & Energy                  63
Manufacturing & Consumer Goods      54
HR & Jobs                           42

In [ ]:
print("FINAL DATASET:", clean.shape)
print("Deals with amount:", clean['Amount_USD'].notna().sum())
print("Total funding ($B):", round(clean['Amount_USD'].sum() / 1e9, 1))
print("\nStages:\n", clean['Stage_Std'].value_counts())
print("\nSectors:\n", clean['Sector'].value_counts())
print("\nRemaining 'Other' industries:\n",
      clean.loc[clean['Sector'] == 'Other', 'Industry'].value_counts().head(10))

FINAL DATASET: (5857, 13)
Deals with amount: 4332
Total funding ($B): 96.7

Stages:
 Stage_Std
Seed/Angel        2196
Private Equity    1214
Unknown/Other     1164
Series A           363
Pre-Series A/B     350
Series B           161
Series C           131
Series D+          124
Pre-Seed            83
Debt                71
Name: count, dtype: int64

Sectors:
 Sector
Tech & SaaS                       1134
Consumer Internet                  848
E-commerce & Retail                716
FinTech                            604
HealthTech                         449
Other                              326
Logistics & Mobility               309
Food & Beverage                    260
Media & Entertainment              231
Unknown                            219
Business Services                  202
EdTech                             166
AgriTech                            98
Real Estate                         71
Travel & Hospitality                65
CleanTech & Energy                  63
Manufac